# Gemma 4 classifier for Enron accounting emails

This notebook replicates the Jev evaluation workflow with the open-weight [Gemma 4 26B A4B IT model](https://openrouter.ai/google/gemma-4-26b-a4b-it) through OpenRouter chat completions. It is zero-shot classification: no weights are trained. The same 559 held-out emails and labels are used; only email bodies are sent to the model. Predictions, source type, and a brief reason are saved after every request.

**News policy:** The default is `exclude`, reflecting the concern that article excerpts create false positives. A news clipping is classified negative even if it discusses Enron wrongdoing. This changes the positive-class definition from the Jev notebook, whose rule includes relevant news. Therefore its metrics should not be read as a direct model-only comparison. Set `NEWS_POLICY = 'include'` to match the Jev rule more closely and use a separate output filename.


## 1. Configure the run

Set `OPENROUTER_API_KEY` in the environment that launches Jupyter; do not save it in the notebook. Set `MAX_TEST_ROWS = 100` for a pilot. `MAX_WORKERS` allows 1–10 simultaneous API requests (threads, since requests wait on network I/O). Requests may incur OpenRouter charges. Completed rows resume without another API call.

Reasoning options: `default` preserves provider defaults, `disabled` turns reasoning off, and `enabled` turns it on with a 8,192-token output cap. Results are saved separately. For comparison against the historical saved inputs, including their verified line endings, use [the reasoning comparison notebook](gemma4_reasoning_options.ipynb).


In [ ]:
import os
import sys
from pathlib import Path
from collections import Counter

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data' / 'enron_labeled_curated_test.csv').exists()), None)
if REPO is None:
    raise FileNotFoundError('Start Jupyter inside the GTPracticum repository')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.gemma4_openrouter import (
    MODEL, MAX_EMAIL_CHARS, NEWS_RULES, evaluate, load_labeled_rows,
    run_classifier, sample_rows,
)

TEST_PATH = REPO / 'data' / 'enron_labeled_curated_test.csv'
NEWS_POLICY = 'exclude'  # 'include' approximates the Jev positive-class definition.
MAX_TEST_ROWS = None  # Set to 100 for a random pilot.
MAX_WORKERS = 10  # 1–10 simultaneous OpenRouter requests.
SEED = 42  # Dataset sampling seed.
GENERATION_SEED = 2  # Sent to OpenRouter on every request.
REASONING_MODE = 'default'  # 'default', 'disabled', or 'enabled'
REASONING_OPTIONS = {'default': (None, '', 350),
                     'disabled': (False, '_no_reasoning', 350),
                     'enabled': (True, '_reasoning_enabled_8192', 8192)}
REASONING_ENABLED, REASONING_SUFFIX, MAX_OUTPUT_TOKENS = REASONING_OPTIONS[REASONING_MODE]
REASONING_SAMPLING = None  # Keep temperature 0 and original sampling defaults.
OUTPUT_PATH = REPO / 'data' / f'gemma4_{NEWS_POLICY}{REASONING_SUFFIX}_seed{GENERATION_SEED}_test_predictions.csv'

all_test_rows = load_labeled_rows(TEST_PATH)
selected_rows = sample_rows(all_test_rows, MAX_TEST_ROWS, seed=SEED)
print('Model:', MODEL)
print('News rule:', NEWS_RULES[NEWS_POLICY])
print('Full test set:', len(all_test_rows), Counter(r['label'] for r in all_test_rows))
print('Selected:', len(selected_rows), Counter(r['label'] for r in selected_rows))
print('Results file:', OUTPUT_PATH)


## 2. Classify

The model receives the email body, source-type rule, and a strict JSON output schema. It returns a discrete label and a short explanation, **not a calibrated probability**. The code clips emails longer than `MAX_EMAIL_CHARS` and records truncation. A configuration hash prevents mixing predictions made with different settings in one CSV.


In [ ]:
results = run_classifier(
    selected_rows, OUTPUT_PATH,
    api_key=os.environ.get('OPENROUTER_API_KEY', ''),
    model=MODEL, max_email_chars=MAX_EMAIL_CHARS, news_policy=NEWS_POLICY,
    max_workers=MAX_WORKERS, reasoning_enabled=REASONING_ENABLED,
    max_output_tokens=MAX_OUTPUT_TOKENS, sampling=REASONING_SAMPLING, seed=GENERATION_SEED,
)
print(f'Classified {len(results)} emails; saved to {OUTPUT_PATH}')


## 3. Evaluate and inspect decisions

Metrics use the existing curated labels, which may treat news differently from the default Gemma rule. The source-type counts and error examples make that disagreement visible. Reasons are model-generated explanations, not independent evidence.


In [ ]:
metrics = evaluate(results)
for name, value in metrics.items():
    print(f'{name}: {value:.4f}' if isinstance(value, float) else f'{name}: {value}')
print('Source types:', Counter(r['source_type'] for r in results))
print('News predictions:', Counter((int(r['label']), int(r['prediction'])) for r in results if r['source_type'] == 'news'))


In [ ]:
errors = [r for r in results if int(r['label']) != int(r['prediction'])]
for r in errors[:20]:
    email = all_test_rows[int(r['source_row'])]['email']
    print(f"row {r['source_row']} | truth={r['label']} pred={r['prediction']} | {r['source_type']}")
    print('Reason:', r['reason'])
    print('Email excerpt:', email[:400].replace('\n', ' '))
    print()
